# Tutorial: Picket-Fence Model Hamiltonian


## Introduction


In this tutorial, we'll focus on the picket-fence model Hamiltonian, a toy model with equally spaced single-particle levels that is widely used in pairing and Richardson–Gaudin (RG) physics. We will show how to construct picket-fence Hamiltonians with the ModelHamiltonian package, generate their integrals, and use them in simple many-body calculations.


## Model definition


The picket-fence model is defined by a set of equally spaced single-particle energy levels

$$

\{\mu_p\}_{p=0}^{N-1}, \qquad

\mu_p = \epsilon_0 + p\Delta

$$

where $\epsilon_0$ is the lowest level energy and $\Delta$ is the constant level spacing.



In its simplest (non-interacting) form, the Hamiltonian is

$$

H_0 = \sum_{p,\sigma} \mu_p \, c^{\dagger}_{p\sigma} c_{p\sigma}

$$

where $c^{\dagger}_{p\sigma}$ and $c_{p\sigma}$ create and annihilate a fermion in level $p$ with spin $\sigma \in \{\uparrow,\downarrow\}$.



To include pairing correlations, one typically considers a reduced BCS / Richardson–Gaudin Hamiltonian of the form

$$

H_{\text{RG}} = \sum_p \mu_p N_p + \sum_{p,q} J_{pq} P_p^{\dagger} P_q

$$

with

$$

N_p = c^{\dagger}_{p\uparrow} c_{p\uparrow} + c^{\dagger}_{p\downarrow} c_{p\downarrow}, \\

P_p^{\dagger} = c^{\dagger}_{p\uparrow} c^{\dagger}_{p\downarrow}, \\

P_p = c_{p\downarrow} c_{p\uparrow}

$$

and $J_{pq}$ controlling the strength of pair scattering between levels $p$ and $q$.



In the `ModelHamiltonian` package, this structure is implemented by the `HamRG` class, which expects the single-particle energies `mu` and the coupling matrix `J_eq` as input. In the examples below we build these objects explicitly for a picket-fence spectrum from the more physical parameters $(\epsilon_0, \Delta, g)$ and also provide a small helper function `make_picket_fence_rg` that automates this construction.



In this tutorial we will:



- Choose a picket-fence spectrum $\{\mu_p\}$,

- Specify a simple “equatorial” coupling matrix $J_{pq} = J_\text{eq}$,

- Use `HamRG(mu=mu, J_eq=J_eq)` to generate the corresponding zero-, one-, and two-body integrals.


In [1]:
# Import Required Libraries for Picket-Fence Model

import numpy as np
import matplotlib.pyplot as plt

from moha.hamiltonians import HamRG
from scipy.linalg import eigh

# Optional: PySCF for FCI calculations (not required for basic usage)

try:
    from pyscf import fci
    HAS_PYSCF = True
except ImportError:
    HAS_PYSCF = False
    print("PySCF not installed; FCI examples will be skipped.")

# Helper function: construct a picket-fence Richardson–Gaudin Hamiltonian
def make_picket_fence_rg(N, epsilon0, spacing, g):
    """Return a HamRG instance for a picket-fence spectrum.
    Parameters
    ----------
    N : int
        Number of single-particle levels.
    epsilon0 : float
        Lowest single-particle energy.
    spacing : float
        Constant level spacing Δ between successive levels.
    g : float
        Pairing interaction strength (we use J_{pq} = -g/2 for all p,q).

    Returns
    -------
    HamRG
        A Richardson–Gaudin Hamiltonian with equally spaced levels and
        constant equatorial coupling.

    """

    indices = np.arange(N)
    mu = epsilon0 + spacing * indices
    J_eq = -0.5 * g * np.ones((N, N))
    return HamRG(mu=mu, J_eq=J_eq)


PySCF not installed; FCI examples will be skipped.


In [2]:
# Defining a 4-site picket-fence Hamiltonian from connectivity tuples

# We model a 4-level picket-fence chain as four sites L1-L4 with nearest-neighbor couplings.
system = [("L1", "L2", 1),
          ("L2", "L3", 1),
          ("L3", "L4", 1)]

# Picket-fence single-particle spectrum parameters
epsilon0 = -1.5  # lowest level energy
spacing = 1.0    # constant level spacing Δ
N = 4
indices = np.arange(N)
mu = epsilon0 + spacing * indices

# For this tutorial, we use the Richardson–Gaudin Hamiltonian (HamRG)
# with J_ax = 0, which includes the picket-fence model as a special case.
# We take a simple constant equatorial coupling J_eq.
J0 = -0.5
J_eq = J0 * np.ones((N, N))

pf_tuples = HamRG(mu=mu, J_eq=J_eq)

print("Number of sites (levels):", pf_tuples.n_sites)
print("Single-particle energies μ_p:", mu)

Number of sites (levels): 4
Single-particle energies μ_p: [-1.5 -0.5  0.5  1.5]


In [3]:
# Defining the same 4-site picket-fence Hamiltonian from a connectivity matrix

# For HamRG we only need the level energies μ and couplings J_eq,
# but here we build a 4x4 adjacency matrix to mirror the Hubbard demonstration style.
connectivity_4 = np.array([[0, 1, 0, 1],
                           [1, 0, 1, 0],
                           [0, 1, 0, 1],
                           [1, 0, 1, 0]])  # 4-site ring

# Reuse the same μ and J_eq as above to define another HamRG instance.
pf_matrix = HamRG(mu=mu, J_eq=J_eq)

# In this simple construction, pf_tuples and pf_matrix are defined
# by the same parameters, so they should generate identical one-body integrals.
h1_tuples = pf_tuples.generate_one_body_integral(dense=True, basis="spatial basis")
h1_matrix = pf_matrix.generate_one_body_integral(dense=True, basis="spatial basis")

print("One-body matrices equal?", np.allclose(h1_tuples, h1_matrix))

One-body matrices equal? True


In [4]:
# Generating integrals for an 8-site picket-fence chain

# Define an 8-site ring connectivity (for illustration; HamRG itself only
# requires μ and J_eq, but we keep the connectivity for analogy with the
# Hubbard tutorial.
connectivity_8 = np.array([[0, 1, 0, 0, 0, 0, 0, 1],
                           [1, 0, 1, 0, 0, 0, 0, 0],
                           [0, 1, 0, 1, 0, 0, 0, 0],
                           [0, 0, 1, 0, 1, 0, 0, 0],
                           [0, 0, 0, 1, 0, 1, 0, 0],
                           [0, 0, 0, 0, 1, 0, 1, 0],
                           [0, 0, 0, 0, 0, 1, 0, 1],
                           [1, 0, 0, 0, 0, 0, 1, 0]])  # 8-site ring

N8 = 8
epsilon0_8 = -3.5
spacing_8 = 1.0
indices_8 = np.arange(N8)
mu_8 = epsilon0_8 + spacing_8 * indices_8

g = -0.25  # interaction strength
J_eq_8 = g * np.ones((N8, N8))

pf_8 = HamRG(mu=mu_8, J_eq=J_eq_8)

e0_8 = pf_8.generate_zero_body_integral()
h1_8 = pf_8.generate_one_body_integral(dense=True, basis="spatial basis")
h2_8 = pf_8.generate_two_body_integral(dense=True, basis="spatial basis", sym=8)

print("Zero-body (core) energy E0:", e0_8)
print("One-body integrals (spatial basis):\n", h1_8)
print("Shape of two-body integrals (spatial basis):", h2_8.shape)

Zero-body (core) energy E0: -1.0
One-body integrals (spatial basis):
 [[-1.625  0.     0.     0.     0.     0.     0.     0.   ]
 [ 0.    -1.125  0.     0.     0.     0.     0.     0.   ]
 [ 0.     0.    -0.625  0.     0.     0.     0.     0.   ]
 [ 0.     0.     0.    -0.125  0.     0.     0.     0.   ]
 [ 0.     0.     0.     0.     0.375  0.     0.     0.   ]
 [ 0.     0.     0.     0.     0.     0.875  0.     0.   ]
 [ 0.     0.     0.     0.     0.     0.     1.375  0.   ]
 [ 0.     0.     0.     0.     0.     0.     0.     1.875]]
Shape of two-body integrals (spatial basis): (8, 8, 8, 8)


In [ ]:
# Exploring pairing strength dependence: ground state energy vs. g

# Scan the ground state energy as a function of the pairing strength g
# to observe how the system's energy is affected by correlation effects.
g_values = np.linspace(-0.5, 0, 20)
ground_state_energies = []

for g_val in g_values:
    J_eq_scan = g_val * np.ones((N8, N8))
    pf_scan = HamRG(mu=mu_8, J_eq=J_eq_scan)
    e0 = pf_scan.generate_zero_body_integral()
    ground_state_energies.append(e0)

plt.figure(figsize=(8, 5))
plt.plot(g_values, ground_state_energies, 'o-', linewidth=2, markersize=6, color='steelblue')
plt.xlabel('Pairing strength g', fontsize=12)
plt.ylabel('Zero-body energy E₀', fontsize=12)
plt.title('Picket-fence model: Effect of pairing on ground state energy', fontsize=13)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Ground state energy range:", f"{min(ground_state_energies):.4f} to {max(ground_state_energies):.4f}")

In [ ]:
# Using the helper function to build a picket-fence RG Hamiltonian
N_test = 6
epsilon0_test = -2.5
spacing_test = 1.0
g_test = 1.0

pf_6 = make_picket_fence_rg(N=N_test, epsilon0=epsilon0_test, spacing=spacing_test, g=g_test)
h1_6 = pf_6.generate_one_body_integral(dense=True, basis="spatial basis")

print("6-level one-body integrals shape:", h1_6.shape)
print("Single-particle energies μ_p (helper):", epsilon0_test + spacing_test * np.arange(N_test))

# Non-interacting picket-fence (g = 0)
N0 = 4
epsilon0_0 = -1.5
spacing_0 = 1.0
g0 = 0.0

pf_nonint = make_picket_fence_rg(N=N0, epsilon0=epsilon0_0, spacing=spacing_0, g=g0)
h1_nonint = pf_nonint.generate_one_body_integral(dense=True, basis="spatial basis")
evals, _ = eigh(h1_nonint)

print("Analytic μ_p:", epsilon0_0 + spacing_0 * np.arange(N0))
print("Eigenvalues of h1_nonint:", np.round(evals, 8))


6-level one-body integrals shape: (6, 6)
Single-particle energies μ_p (helper): [-2.5 -1.5 -0.5  0.5  1.5  2.5]
Analytic μ_p: [-1.5 -0.5  0.5  1.5]
Eigenvalues of h1_nonint: [-0.75 -0.25  0.25  0.75]
